In [1]:
%pip install torch numpy tensorboard

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install torchview graphviz

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter

#folders
for folder in ["data","checkpoints", "runs"]:
    os.makedirs(folder, exist_ok=True)


In [4]:
#data path
train_data_path="./data/mnist_train.csv"
test_data_path="./data/mnist_test.csv"

# files={"data/mnist_train.csv":"./data/mnist_train.csv",
#        "data/mnist_test.csv":"./data/mnist_test.csv"}


In [ ]:
#loading the data
#read a CSV into PyTorch tesnors with right shape and dtype
IMAGE_SIZE=28

def load_csv(data_path):
    """input:csv data path. each row in CSV: label, IMG_SIZE*IMG_SIZE pixels
        output: return pixels and labels as separate Numpy arrays
    """
    
    #raw data
    raw=np.genfromtxt(data_path, delimiter=",", dtype=np.float32)
    
    #column 0 is the label. 
    labels=raw[:,0].astype(np.int64)

    pixels=raw[:,1:]

    return pixels, labels

def to_tensors(pixels, labels, normalize=True):
    """convert NumPy pixel and label arrays into PyTorch tensors
    PyTorch tensor dimension (N,C,H,W) where (N=batch size, C=channels, H=height, W=width)
    """

    if normalize:
        #scaling pixels from 0-255 to 0-1
        pixels=pixels/255.0
    
    #-1 automatically is replaced by a proper value from Numpy
    print("#####")
    print("Before:",pixels.shape, pixels.dtype)
    pixels=pixels.reshape(-1,1,IMAGE_SIZE,IMAGE_SIZE)

    print("After:",pixels.shape, pixels.dtype)
    print("#####")
    #manually performing PyTorch transforamtions steps instead of calling direct function
    #genfromtxt, reshape, np.ascontiguousarray, from_numpy to tensors, float32
    # PyTorch equivalent
    # from torchvision import datasets, transforms
    # ds = datasets.MNIST(root=".", download=True, transform=transforms.ToTensor())
    x=torch.from_numpy(np.ascontiguousarray(pixels)).float()
    y=torch.from_numpy(labels)

    return x,y

def load_train_data(data_path, validation_size=500):
    pixels,labels=load_csv(data_path)
    split=len(pixels)-validation_size

    x_train,y_train=to_tensors(pixels[:split],labels[:split])
    x_val,y_val=to_tensors(pixels[split:],labels[split:])

    return x_train, x_val, y_train, y_val

def load_test_data(data_path):
    pixels, labels=load_csv(data_path)
    return to_tensors(pixels,labels)



In [6]:
#checking shapes
x_train, x_val, y_train, y_val= load_train_data(train_data_path)
print(f"train inputs: {tuple(x_train.shape)}, {x_train.dtype}")
print(f"train labels: {tuple(y_train.shape)}, {y_train.dtype}")
print(f"val inputs: {tuple(x_val.shape)}, {x_val.dtype}")
print(f"val labels: {tuple(y_val.shape)}, {y_val.dtype}")

print("pixel range:", float(x_train.min()), float(x_train.max()))

#####
Before: (59500, 784) float32
After: (59500, 1, 28, 28) float32
#####
#####
Before: (500, 784) float32
After: (500, 1, 28, 28) float32
#####
train inputs: (59500, 1, 28, 28), torch.float32
train labels: (59500,), torch.int64
val inputs: (500, 1, 28, 28), torch.float32
val labels: (500,), torch.int64
pixel range: 0.0 1.0


In [ ]:
#defining the model


[0 1 2 3 4 5 6 7 8 9]


In [7]:
#raw=np.genfromtxt("./data/mnist_train.csv", delimiter=",", dtype=np.float32)


In [11]:
torch.cuda.is_available()

/home/mitochondria/Desktop/Learning/deep-learning-cnn/.dlcnnvenv/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


False